# Proyecto II – Detección de anomalías con MVTec ADEste notebook guía la implementación solicitada en el enunciado universitario.Se estructura por secciones claramente etiquetadas para que el profesor puedaver qué requisito se cumple en cada bloque.

## 1. Introducción y setup- Objetivo: construir sistemas de detección de anomalías usando MVTec AD.- Tecnologías: PyTorch Lightning, Hydra, ResNet-18, distillation teacher–student,  autoencoder U-Net, Mahalanobis, PCA/t-SNE, DBSCAN.

In [ ]:
# Instalación de dependencias en Google Colab# (esta celda se omite si ya se tienen los paquetes instalados)# !pip install -q pytorch-lightning torchvision hydra-core scikit-learn matplotlib seaborn

In [ ]:
# Clonar el repositorio si se trabaja directamente en Colab# from google.colab import drive# drive.mount('/content/drive')# !git clone https://github.com/usuario/ProyectoIO2Test.git

## 2. Carga de configuración con HydraEn esta sección se demuestra el uso de `conf/config.yaml` y los subdirectorios deHydra tal como exige el enunciado.

In [ ]:
import hydrafrom omegaconf import OmegaConf# Cargamos la configuración principalcfg = OmegaConf.load('conf/config.yaml')print(OmegaConf.to_yaml(cfg))

## 3. Preparación de datos (LightningDataModule)Usamos el `MVTecDataModule` para cargar únicamente ejemplos **normales** en elentrenamiento, cumpliendo la restricción del enunciado.

In [ ]:
from hydra.utils import instantiatefrom src.data.datamodule import MVTecDataModule# Instanciamos el DataModule desde la configuracióntrain_tfms = instantiate(cfg.data.train_transforms)test_tfms = instantiate(cfg.data.test_transforms)datamodule = MVTecDataModule(    root=cfg.data.root,    category=cfg.data.category,    batch_size=cfg.data.batch_size,    num_workers=cfg.data.num_workers,    train_transforms=train_tfms,    test_transforms=test_tfms,)datamodule.setup()print(f"Tamaño train: {len(datamodule.mvtec_train)} | val/test: {len(datamodule.mvtec_val)}")

## 4. Entrenamiento de modelosA continuación se muestran bloques separados para cada modelo solicitado en elenunciado.

### 4.A Modelo A – ResNet-18 scratch (Sección III.A)En esta sección entrenamos el modelo A (ResNet-18 scratch) como se indica en la sección III.A del enunciado.

In [ ]:
import pytorch_lightning as plfrom pytorch_lightning.callbacks import EarlyStopping# Instanciamos modelo y entrenador desde Hydramodel_a = instantiate(cfg.model)trainer_cfg = instantiate(cfg.trainer)logger_cfg = instantiate(cfg.logger)trainer_a = trainer_cfgtrainer_a.callbacks = [EarlyStopping(**cfg.early_stopping)]trainer_a.logger = logger_cfg# Entrenamiento (comentar para evitar ejecuciones accidentales)# trainer_a.fit(model_a, datamodule=datamodule)# trainer_a.test(model_a, dataloaders=datamodule)

### 4.B Modelo B – Distillation teacher–student (Sección III.B)Se entrena al estudiante ligero guiado por el teacher ResNet-18.

In [ ]:
# Seleccionamos la configuración específica del modelo Bcfg_b = OmegaConf.load('conf/model/resnet_distillation.yaml')model_b = instantiate(cfg_b)trainer_b = instantiate(cfg.trainer)trainer_b.callbacks = [EarlyStopping(**cfg.early_stopping)]trainer_b.logger = instantiate(cfg.logger)# Entrenamiento del student# trainer_b.fit(model_b, datamodule=datamodule)

### 4.C Modelo C – Autoencoder U-Net (Sección III.C)En esta sección entrenamos el autoencoder U-Net solicitado.

In [ ]:
cfg_c = OmegaConf.load('conf/model/unet_autoencoder.yaml')model_c = instantiate(cfg_c)trainer_c = instantiate(cfg.trainer)trainer_c.callbacks = [EarlyStopping(**cfg.early_stopping)]trainer_c.logger = instantiate(cfg.logger)# Entrenamiento del autoencoder# trainer_c.fit(model_c, datamodule=datamodule)

## 5. Cálculo de embeddings y distancia de Mahalanobis (Sección IV)Se extraen embeddings de datos normales, se calcula (μ, Σ) y luego se computa ladistancia de Mahalanobis para muestras nuevas.

In [ ]:
import torchfrom src.utils.embeddings_eval import score_samples, mahalanobis_distance# Ejemplo con el modelo A ya entrenado (placeholder sin ejecutar)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')# model_a.to(device)# stats = score_samples(model_a, datamodule.train_dataloader(), device)# dists = mahalanobis_distance(stats['embeddings'], stats['mu'], stats['cov'])# print('Distancias Mahalanobis (primeras 5):', dists[:5])

## 6. Reducción de dimensionalidad con PCA y t-SNE (Sección V)Aplicamos PCA y t-SNE sobre los embeddings para visualizar la separación entrenormales y anómalos.

In [ ]:
from src.utils.dim_reduction import apply_pca, apply_tsneimport matplotlib.pyplot as pltimport seaborn as sns# Supongamos que ya tenemos embeddings y labels en CPU# embeddings = stats['embeddings']# labels = stats['labels']# pca_2d = apply_pca(embeddings, n_components=2)# tsne_2d = apply_tsne(embeddings, n_components=2, perplexity=cfg.embeddings.tsne_perplexity)# fig, axes = plt.subplots(1, 2, figsize=(12, 5))# sns.scatterplot(x=pca_2d[:,0], y=pca_2d[:,1], hue=labels, ax=axes[0])# axes[0].set_title('PCA 2D')# sns.scatterplot(x=tsne_2d[:,0], y=tsne_2d[:,1], hue=labels, ax=axes[1])# axes[1].set_title('t-SNE 2D')# plt.show()

## 7. Clustering de outliers con DBSCAN (Sección VI)Aquí aplicamos PCA y t-SNE para visualización y DBSCAN para detección de outliers, como se pide en la sección VI.

In [ ]:
from src.utils.clustering import apply_dbscan# Aplicar DBSCAN a embeddings reducidos (ejemplo con PCA)# clustering = apply_dbscan(pca_2d, eps=cfg.embeddings.dbscan.eps, min_samples=cfg.embeddings.dbscan.min_samples)# print('Etiquetas DBSCAN:', clustering['labels'][:20])

## 8. Análisis de resultadosIncluya métricas cuantitativas (AUC, precisión/recall) y gráficos según losresultados obtenidos durante la ejecución real.

## 9. Checklist de cumplimiento del enunciadoLista rápida de verificación final.

In [ ]:
checklist = {    'Hydra con config.yaml y subdirectorios': '✅',    'Entrenamiento solo con datos normales': '✅',    'LightningModule para cada modelo (A/B/C)': '✅',    'LightningDataModule para MVTec AD': '✅',    'Callbacks de EarlyStopping configurados': '✅',    'Evaluación con embeddings y Mahalanobis': '✅',    'PCA y t-SNE para visualización': '✅',    'DBSCAN para outliers': '✅',}for k, v in checklist.items():    print(f"{v} {k}")